# Notebook 4 - Fake Engagement Detection

trying to find suspicious users who might be artificially inflating or lowering ratings

In [ ]:
# imports
import os
os.makedirs('reports', exist_ok=True)
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
np.random.seed(42)

print('libraries ready')

## Load Data

In [ ]:
# load csv files
ratings = pd.read_csv('data/ratings.csv', nrows=100000, dtype={
    'userId':  'int32',
    'movieId': 'int32',
    'rating':  'float32'
})
movies  = pd.read_csv('data/movies.csv', dtype={
    'movieId': 'int32'
})

print(f'Ratings shape : {ratings.shape}')
print(ratings.head())

## Build User Features

computing 4 features per user - total ratings, avg rating, std deviation and how many extreme ratings they gave

In [ ]:
# per user stats

user_features = ratings.groupby('userId').agg(
    total_ratings=('rating', 'count'),
    avg_rating   =('rating', 'mean'),
    rating_std   =('rating', 'std')
).reset_index()

# checking extreme ratings
is_extreme = ratings['rating'].isin([1.0, 5.0]).astype(int)
extreme_pct = ratings.assign(is_extreme=is_extreme).groupby('userId')['is_extreme'].mean().reset_index()
extreme_pct.columns = ['userId', 'pct_extreme']

user_features = pd.merge(user_features, extreme_pct, on='userId')

# fillna for users who gave only 1 rating
user_features['rating_std'] = user_features['rating_std'].fillna(0)

print(f'User features shape: {user_features.shape}')
print(user_features.describe().round(3))

## Visualise User Features

In [ ]:
# Plot histograms for all four features
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

feature_info = [
    ('total_ratings', 'Total Ratings per User',        'Number of Ratings', 'steelblue'),
    ('avg_rating',    'Average Rating per User',       'Average Rating',    'coral'),
    ('rating_std',    'Rating Std Deviation per User', 'Std Deviation',     'seagreen'),
    ('pct_extreme',   '% Extreme Ratings (1 or 5)',    'Fraction (0-1)',    'mediumpurple'),
]

for ax, (col, title, xlabel, color) in zip(axes.flat, feature_info):
    ax.hist(user_features[col], bins=40, color=color, alpha=0.8, edgecolor='white')
    ax.set_title(title, fontsize=12)
    ax.set_xlabel(xlabel, fontsize=10)
    ax.set_ylabel('Number of Users', fontsize=10)

plt.suptitle('User Behaviour Feature Distributions', fontsize=15, y=1.02)
plt.tight_layout()
plt.savefig('reports/user_features.png', dpi=80)
plt.show()
plt.close('all')
print('Plot saved!')

## Isolation Forest

isolation forest works by randomly splitting data - anomalous points get isolated faster because theyre far from the main cluster. setting contamination to 0.05 so it flags roughly 5% as suspicious

In [ ]:
# Features for anomaly detection
feature_cols = ['total_ratings', 'avg_rating', 'rating_std', 'pct_extreme']
X = user_features[feature_cols].values

# scale so no feature dominates
scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train Isolation Forest
iso_forest = IsolationForest(
    contamination=0.05,   # expect 5% suspicious users
    random_state=42,
    n_estimators=50
)
predictions = iso_forest.fit_predict(X_scaled)

# Isolation Forest returns: -1 = anomaly, +1 = normal
user_features['label']     = predictions
user_features['is_suspect'] = (predictions == -1)   # True = suspicious
user_features['status']    = user_features['is_suspect'].map({True: 'Suspicious', False: 'Normal'})

# Anomaly score: more negative = more anomalous
user_features['anomaly_score'] = iso_forest.score_samples(X_scaled)

num_suspicious = user_features['is_suspect'].sum()
total_users    = len(user_features)
print(f'Total users     : {total_users:,}')
print(f'Suspicious users: {num_suspicious:,} ({num_suspicious/total_users*100:.1f}%)')
print(f'Normal users    : {total_users - num_suspicious:,}')

## Visualise Results

In [ ]:
# scatter plot - avg rating vs total ratings
plt.figure(figsize=(12, 6))

normal    = user_features[user_features['status'] == 'Normal']
suspicious = user_features[user_features['status'] == 'Suspicious']

plt.scatter(normal['total_ratings'],    normal['avg_rating'],
            c='steelblue', alpha=0.4, s=15, label='Normal Users')
plt.scatter(suspicious['total_ratings'], suspicious['avg_rating'],
            c='red', alpha=0.8, s=40, label='Suspicious Users', zorder=5)

plt.xscale('log')   # log scale because activity range is huge
plt.title('User Anomaly Detection — Average Rating vs Total Ratings', fontsize=14)
plt.xlabel('Total Ratings (log scale)', fontsize=12)
plt.ylabel('Average Rating Given', fontsize=12)
plt.legend(fontsize=11)
plt.tight_layout()
plt.savefig('reports/anomaly_scatter.png', dpi=80)
plt.show()
plt.close('all')

# bar chart of counts
counts = user_features['status'].value_counts()
plt.figure(figsize=(6, 4))
sns.barplot(x=counts.index, y=counts.values, 
            palette={'Normal': 'steelblue', 'Suspicious': 'red'})
plt.title('Normal vs Suspicious User Count', fontsize=13)
plt.xlabel('User Status', fontsize=11)
plt.ylabel('Number of Users', fontsize=11)
for i, v in enumerate(counts.values):
    plt.text(i, v + 5, str(v), ha='center', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('reports/suspicious_count.png', dpi=80)
plt.show()
plt.close('all')

## Top 10 Most Suspicious Users

In [ ]:
# Sort by anomaly score (most negative = most suspicious)
top_suspicious = (user_features[user_features['is_suspect']]
                  .sort_values('anomaly_score')
                  .head(10)[['userId', 'total_ratings', 'avg_rating',
                              'rating_std', 'pct_extreme', 'anomaly_score']])

print('Top 10 Most Suspicious Users:')
print(top_suspicious.round(3).to_string(index=False))

## Movies with Most Suspicious Raters

In [ ]:
# Mark each rating as suspicious or not
suspect_userids = set(user_features[user_features['is_suspect']]['userId'])
ratings['is_suspicious'] = ratings['userId'].isin(suspect_userids)

# Per movie: count total raters and suspicious raters
movie_suspect = ratings.groupby('movieId').agg(
    total_raters    =('userId', 'count'),
    suspicious_raters=('is_suspicious', 'sum')
).reset_index()

movie_suspect['pct_suspicious'] = (
    movie_suspect['suspicious_raters'] / movie_suspect['total_raters'] * 100
).round(2)

# merge in movie titles
movie_suspect = pd.merge(movie_suspect, movies[['movieId', 'title']], on='movieId')

# Sort by % suspicious (only consider movies with 10+ raters to avoid tiny movies)
top_suspect_movies = (movie_suspect[movie_suspect['total_raters'] >= 10]
                      .sort_values('pct_suspicious', ascending=False)
                      .head(15))

print('Movies with Most Suspicious Raters (min 10 raters):')
print(top_suspect_movies[['title', 'total_raters', 'suspicious_raters', 'pct_suspicious']]
      .to_string(index=False))

# Bar chart
plt.figure(figsize=(13, 6))
sns.barplot(data=top_suspect_movies, x='pct_suspicious', y='title', palette='Reds_r')
plt.title('Top 15 Movies with Highest % Suspicious Raters', fontsize=14)
plt.xlabel('% Suspicious Raters', fontsize=12)
plt.ylabel('Movie Title', fontsize=12)
plt.tight_layout()
plt.savefig('reports/suspicious_movies.png', dpi=80)
plt.show()
plt.close('all')

print('\ndone')